# 📚 Sampling & Confidence Intervals: Honest Uncertainty
> Section 4 of the Statistics & A/B Testing curriculum  
> 통계 & A/B 테스트 커리큘럼 섹션 4

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Compute standard error and explain why it shrinks with the square root of sample size<br>표준오차를 계산하고 표본 크기의 제곱근에 반비례해 줄어드는 이유 설명하기
- [x] Build a confidence interval for a mean (t-distribution) and for a proportion (Normal)<br>평균(t-분포)과 비율(정규분포)의 신뢰구간 만들기
- [x] Correctly interpret what "95% confidence" means, avoiding the most common statistical misconception<br>"95% 신뢰"가 무엇을 의미하는지 올바르게 해석하고, 가장 흔한 통계적 오해 피하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** A sample statistic (like a sample mean) varies from sample to sample — if you took 100 different random samples of customers, you'd get 100 slightly different average order values. A confidence interval captures that variability into a single range: "we're 95% confident the TRUE population average falls between $X and $Y," instead of pretending a single sample gives you the exact truth.

**KR:** 표본 통계량(표본평균 등)은 표본마다 달라집니다 -- 고객의 무작위 표본 100개를 뽑으면 100개의 조금씩 다른 평균 주문금액을 얻게 됩니다. 신뢰구간은 이 변동성을 하나의 범위로 담아냅니다: 단일 표본이 정확한 진실을 준다고 가정하는 대신, "실제 모집단 평균이 $X와 $Y 사이에 있다고 95% 확신한다"고 말합니다.

### CI for a mean vs. CI for a proportion (평균 vs 비율의 신뢰구간)

| | CI for a Mean (평균) | CI for a Proportion (비율) |
|---|---|---|
| Formula / 공식 | x̄ ± t* × (s/√n) | p̂ ± z* × √(p̂(1-p̂)/n) |
| Distribution used / 사용 분포 | t-distribution (extra uncertainty from estimating σ)<br>t-분포(σ 추정의 추가 불확실성 반영) | Normal, via CLT (a proportion is a binomial mean)<br>정규분포, CLT 경유(비율은 이항분포의 평균) |
| scipy function / scipy 함수 | `stats.t.interval()` | manual, with `stats.norm.ppf()` |
| Example / 예시 | Average order value<br>평균 주문금액 | Conversion rate<br>전환율 |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** Reporting "average order value: $52" as if it's an exact, permanent fact is misleading — a different sample from the same population might give $49 or $55. Reporting "average order value: $52, 95% CI [$48, $56]" tells the truth: you have a good estimate, but there's honest uncertainty around it, and the width of that uncertainty depends directly on how much data you collected.

**KR:** "평균 주문금액: $52"를 마치 정확하고 영구적인 사실인 것처럼 보고하는 것은 오해를 부릅니다 -- 같은 모집단에서 다른 표본을 뽑으면 $49나 $55가 나올 수 있습니다. "평균 주문금액: $52, 95% CI [$48, $56]"라고 보고하는 것이 진실을 말합니다: 좋은 추정치를 가지고 있지만 그 주변에 정직한 불확실성이 있으며, 그 불확실성의 폭은 수집한 데이터의 양에 직접적으로 좌우됩니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Every "average X is Y" claim in a business report should ideally come with a confidence interval, A/B test results always need a CI to judge whether an observed lift is real or noise, and survey results (e.g., "68% of customers are satisfied") need a margin of error to be interpreted correctly.

**KR:** 비즈니스 보고서의 모든 "평균 X는 Y" 주장은 이상적으로는 신뢰구간과 함께 제시되어야 하며, A/B 테스트 결과는 관측된 상승이 실제인지 노이즈인지 판단하기 위해 항상 신뢰구간이 필요하고, 설문 결과(예: "고객의 68%가 만족")는 올바르게 해석되려면 오차범위가 필요합니다.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

np.random.seed(42)
sample = np.random.normal(loc=52, scale=15, size=100)  # a sample of 100 order amounts

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)  # SAMPLE std dev (see Section 1's ddof mistake!)
n = len(sample)
standard_error = sample_std / np.sqrt(n)

print(f"Sample mean: ${sample_mean:.2f} / 표본평균: ${sample_mean:.2f}")
print(f"Standard error: ${standard_error:.2f} / 표준오차: ${standard_error:.2f}")

Sample mean: $50.44 / 표본평균: $50.44
Standard error: $1.36 / 표준오차: $1.36


## Common Variations (자주 쓰는 변형)

In [2]:
# 95% confidence interval using the t-distribution
# t-분포를 사용한 95% 신뢰구간
confidence_level = 0.95
degrees_freedom = n - 1
ci = stats.t.interval(confidence_level, degrees_freedom, loc=sample_mean, scale=standard_error)

print(f"95% CI: [${ci[0]:.2f}, ${ci[1]:.2f}]")

# scipy also lets you get the critical t-value directly
# 임계 t값을 직접 얻을 수도 있음
t_critical = stats.t.ppf(0.975, degrees_freedom)  # for a 2-sided 95% CI
margin_of_error = t_critical * standard_error
print(f"Margin of error: ${margin_of_error:.2f}")

95% CI: [$47.74, $53.15]
Margin of error: $2.70


---
# 🧪 Small Examples (예제)

## Example 1 — Standard Error: How Uncertainty Shrinks with Sample Size (표본 크기에 따른 불확실성 감소)

In [3]:
# Business question: how much does collecting MORE survey responses actually
# reduce our uncertainty about the true average satisfaction score?
# 비즈니스 질문: 설문 응답을 더 모으는 것이 실제 평균 만족도 점수에 대한
# 불확실성을 얼마나 줄여주는가?

np.random.seed(1)
population = np.random.normal(loc=4.2, scale=0.8, size=100000)  # true population

for n in [10, 50, 100, 500, 2000]:
    sample = np.random.choice(population, size=n, replace=False)
    se = sample.std(ddof=1) / np.sqrt(n)
    print(f"n={n:>5}: standard error = {se:.4f}")

print()
print("Standard error shrinks with the SQUARE ROOT of n -- quadrupling your sample")
print("size only halves your uncertainty, not quarters it. Diminishing returns kick in fast.")
print("표준오차는 n의 제곱근에 반비례해 줄어듭니다 -- 표본 크기를 4배로 늘려도")
print("불확실성은 절반만 줄어들지, 4분의 1로 줄지 않습니다. 수확체감이 빠르게 나타납니다.")

n=   10: standard error = 0.2683
n=   50: standard error = 0.1019
n=  100: standard error = 0.0818
n=  500: standard error = 0.0352
n= 2000: standard error = 0.0179

Standard error shrinks with the SQUARE ROOT of n -- quadrupling your sample
size only halves your uncertainty, not quarters it. Diminishing returns kick in fast.
표준오차는 n의 제곱근에 반비례해 줄어듭니다 -- 표본 크기를 4배로 늘려도
불확실성은 절반만 줄어들지, 4분의 1로 줄지 않습니다. 수확체감이 빠르게 나타납니다.


## Example 2 — Confidence Interval for a Mean: Average Order Value (평균 주문금액의 신뢰구간)

In [ ]:
# Business question: report last month's average order value with honest
# uncertainty, not just a single number.
# 비즈니스 질문: 지난달 평균 주문금액을 단일 숫자가 아니라 정직한
# 불확실성과 함께 보고하기.

np.random.seed(2)
orders = np.random.gamma(shape=3, scale=18, size=150)

n = len(orders)
mean = orders.mean()
se = orders.std(ddof=1) / np.sqrt(n)
ci_95 = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)

print(f"Sample size: {n}")
print(f"Average order value: ${mean:.2f}")
print(f"95% CI: [${ci_95[0]:.2f}, ${ci_95[1]:.2f}]")
print()
print("We're 95% confident the TRUE average order value (if we could measure every")
print(f"single order) falls between ${ci_95[0]:.2f} and ${ci_95[1]:.2f}.")
print(f"실제 평균 주문금액(모든 주문을 다 측정할 수 있다면)이 ${ci_95[0]:.2f}와")
print(f"${ci_95[1]:.2f} 사이에 있다고 95% 확신합니다.")

Sample size: 150
Average order value: $53.77
95% CI: [$47.87, $59.67]

We're 95% confident the TRUE average order value (if we could measure every
single order) falls between $47.87 and $59.67.
실제 평균 주문금액(모든 주문을 다 측정할 수 있다면)이 $47.87와
$59.67 사이에 있다고 95% 확신합니다.


## Example 3 — Confidence Interval for a Proportion: Conversion Rate (전환율의 신뢰구간)

In [ ]:
# Business question: is our observed 8% conversion rate precise enough to
# plan a budget around, or could the true rate be meaningfully different?
# 비즈니스 질문: 관측된 8% 전환율이 예산을 계획할 만큼 정밀한가, 아니면
# 실제 전환율이 의미 있게 다를 수 있는가?

n_visitors = 500
n_converted = 40
p_hat = n_converted / n_visitors

se_prop = np.sqrt(p_hat * (1 - p_hat) / n_visitors)
z_critical = stats.norm.ppf(0.975)  # for a 2-sided 95% CI
margin = z_critical * se_prop
ci_lower, ci_upper = p_hat - margin, p_hat + margin

print(f"Observed conversion rate: {p_hat:.1%}")
print(f"Standard error: {se_prop:.4f}")
print(f"95% CI: [{ci_lower:.1%}, {ci_upper:.1%}]")
print()
print(f"The true conversion rate could plausibly be anywhere from {ci_lower:.1%} to")
print(f"{ci_upper:.1%} -- a wide enough range that budget planning should account for")
print(f"the low end, not just the point estimate of {p_hat:.1%}.")
print(f"실제 전환율은 {ci_lower:.1%}에서 {ci_upper:.1%} 사이 어디든 될 수 있습니다 --")
print(f"충분히 넓은 범위이므로 예산 계획은 점추정치 {p_hat:.1%}뿐 아니라 하한도")
print("고려해야 합니다.")

Observed conversion rate: 8.0%
Standard error: 0.0121
95% CI: [5.6%, 10.4%]

The true conversion rate could plausibly be anywhere from 5.6% to
10.4% -- a wide enough range that budget planning should account for
the low end, not just the point estimate of 8.0%.
실제 전환율은 5.6%에서 10.4% 사이 어디든 될 수 있습니다 --
충분히 넓은 범위이므로 예산 계획은 점추정치 8.0%뿐 아니라 하한도
고려해야 합니다.


## Example 4 — What "95% Confidence" Actually Means (95% 신뢰가 실제로 의미하는 것)

In [6]:
# Business question: what does "95% confidence" actually mean? Let's find out
# by simulating the process 200 times and seeing how often the interval
# actually captures the TRUE population mean.
# 비즈니스 질문: "95% 신뢰"는 실제로 무엇을 의미하는가? 이 과정을 200번
# 시뮬레이션해서 구간이 실제 모집단 평균을 얼마나 자주 포함하는지 확인해보자.

np.random.seed(3)
true_mean = 50
true_std = 15
population = np.random.normal(true_mean, true_std, 200000)

n_simulations = 200
sample_size = 40
captured = 0

for _ in range(n_simulations):
    sample = np.random.choice(population, size=sample_size, replace=False)
    s_mean = sample.mean()
    s_se = sample.std(ddof=1) / np.sqrt(sample_size)
    lo, hi = stats.t.interval(0.95, df=sample_size - 1, loc=s_mean, scale=s_se)
    if lo <= true_mean <= hi:
        captured += 1

print(f"True population mean: {true_mean}")
print(f"Out of {n_simulations} independently drawn samples, each with its own 95% CI:")
print(f"{captured} intervals ({captured / n_simulations:.1%}) actually contained the true mean")
print(f"{n_simulations - captured} intervals ({(n_simulations - captured) / n_simulations:.1%}) missed it")
print()
print("THIS is what '95% confidence' means: if you repeated this sampling process many")
print("times, about 95% of the resulting intervals would capture the true value. It does")
print("NOT mean 'there's a 95% probability the true mean is in THIS ONE interval' -- any")
print("single interval either does or doesn't contain the true mean; we just don't know which.")
print()
print("이것이 '95% 신뢰'가 실제로 의미하는 바입니다: 이 표본추출 과정을 여러 번")
print("반복하면 약 95%의 결과 구간이 실제 값을 포함합니다. '이 하나의 구간에 실제")
print("평균이 있을 확률이 95%다'라는 뜻이 아닙니다 -- 하나의 구간은 실제 평균을")
print("포함하거나 포함하지 않거나 둘 중 하나이며, 우리는 어느 쪽인지 알 수 없을 뿐입니다.")

True population mean: 50
Out of 200 independently drawn samples, each with its own 95% CI:
189 intervals (94.5%) actually contained the true mean
11 intervals (5.5%) missed it

THIS is what '95% confidence' means: if you repeated this sampling process many
times, about 95% of the resulting intervals would capture the true value. It does
NOT mean 'there's a 95% probability the true mean is in THIS ONE interval' -- any
single interval either does or doesn't contain the true mean; we just don't know which.

이것이 '95% 신뢰'가 실제로 의미하는 바입니다: 이 표본추출 과정을 여러 번
반복하면 약 95%의 결과 구간이 실제 값을 포함합니다. '이 하나의 구간에 실제
평균이 있을 확률이 95%다'라는 뜻이 아닙니다 -- 하나의 구간은 실제 평균을
포함하거나 포함하지 않거나 둘 중 하나이며, 우리는 어느 쪽인지 알 수 없을 뿐입니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [10]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: standard error -- delivery time uncertainty ----------
np.random.seed(4)
delivery_times = np.random.normal(3.5, 1.2, 80)  # days
n_a = len(delivery_times)

sample_std_method_a = 'std'   # TODO: which pandas Series method computes SAMPLE std dev (ddof=1)? ('std')

if ready(sample_std_method_a):
    std_val = getattr(pd.Series(delivery_times), sample_std_method_a)()
    se_a = std_val / np.sqrt(n_a)
    print(f"Standard error: {se_a:.4f} days")
else:
    print("Part A \u270f\ufe0f  Fill in sample_std_method_a above, then re-run.")

# ---------- Part B: CI for a mean -- customer support rating ----------
np.random.seed(5)
ratings = np.random.normal(4.1, 0.6, 60)

confidence_b = 0.95   # TODO: confidence level as a decimal, e.g. 0.95

if ready(confidence_b):
    n_b = len(ratings)
    mean_b = ratings.mean()
    se_b = ratings.std(ddof=1) / np.sqrt(n_b)
    ci_b = stats.t.interval(confidence_b, df=n_b - 1, loc=mean_b, scale=se_b)
    print(f"{confidence_b:.0%} CI: [{ci_b[0]:.2f}, {ci_b[1]:.2f}]")
else:
    print("Part B \u270f\ufe0f  Fill in confidence_b above, then re-run.")

# ---------- Part C: CI for a proportion -- trial-to-paid upgrade rate ----------
n_trials_c = 300
n_upgraded_c = 45

p_hat_c = n_upgraded_c / n_trials_c   # TODO: compute the observed proportion (n_upgraded_c / n_trials_c)

if ready(p_hat_c):
    se_c = np.sqrt(p_hat_c * (1 - p_hat_c) / n_trials_c)
    z_c = stats.norm.ppf(0.975)
    margin_c = z_c * se_c
    print(f"p_hat = {p_hat_c:.3f}, 95% CI: [{p_hat_c - margin_c:.3f}, {p_hat_c + margin_c:.3f}]")
else:
    print("Part C \u270f\ufe0f  Fill in p_hat_c above, then re-run.")

# ---------- Part D: interpretation check ----------
interpretation_d = False   # TODO: True or False -- "a 95% CI means there's a 95% chance
                           # the true mean is inside THIS specific interval"

if ready(interpretation_d):
    print(f"Your answer: {interpretation_d}")
    if interpretation_d is False:
        print("Correct! A CI is about the long-run success rate of the PROCEDURE, not a")
        print("probability statement about one specific interval.")
        print("정답입니다! 신뢰구간은 절차의 장기적 성공률에 관한 것이지, 특정 하나의")
        print("구간에 대한 확률 진술이 아닙니다.")
    else:
        print("Not quite -- review Example 4's simulation to see why this is a common")
        print("misconception.")
        print("다시 확인해보세요 -- 예제 4의 시뮬레이션에서 이것이 왜 흔한 오해인지 보세요.")
else:
    print("Part D \u270f\ufe0f  Fill in interpretation_d above (True or False), then re-run.")

Standard error: 0.1307 days
95% CI: [4.04, 4.32]
p_hat = 0.150, 95% CI: [0.110, 0.190]
Your answer: False
Correct! A CI is about the long-run success rate of the PROCEDURE, not a
probability statement about one specific interval.
정답입니다! 신뢰구간은 절차의 장기적 성공률에 관한 것이지, 특정 하나의
구간에 대한 확률 진술이 아닙니다.


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Misinterpreting What "95% Confidence" Means (95% 신뢰의 의미 오해)
"There's a 95% probability the true mean is in this interval" sounds right but is technically wrong — the true mean is a fixed number, not a random variable, so it either IS or ISN'T in any specific interval. The 95% describes the long-run success rate of the CI-building PROCEDURE (see Example 4).  
"실제 평균이 이 구간에 있을 확률이 95%다"는 맞는 말처럼 들리지만 엄밀히는 틀렸습니다 -- 실제 평균은 확률변수가 아니라 고정된 숫자이므로, 특정 구간에 있거나 없거나 둘 중 하나입니다. 95%는 신뢰구간을 만드는 절차의 장기적 성공률을 설명합니다(예제 4 참고).

✅ **Fix / 해결:** Say "we're 95% confident" (describing your trust in the method) rather than "there's a 95% probability" (which implies randomness in something that's actually fixed).

### Mistake 2 — Using the Normal (z) Distribution Instead of t for Small Samples (작은 표본에 t 대신 정규(z)분포 사용)
For small samples (roughly n < 30), the sample standard deviation is itself an imprecise estimate of the true population standard deviation — the t-distribution has fatter tails to account for that EXTRA uncertainty. Using z instead makes your CI falsely narrow.  
작은 표본(대략 n<30)에서는 표본표준편차 자체가 실제 모집단 표준편차에 대한 부정확한 추정치입니다 -- t-분포는 그 추가적인 불확실성을 반영하기 위해 더 두꺼운 꼬리를 가집니다. z를 대신 쓰면 신뢰구간이 실제보다 좁게 나옵니다.

✅ **Fix / 해결:** Use `stats.t.interval()` for means (especially with smaller samples); z is only appropriate when the population standard deviation is truly known, which is rare in business data.

### Mistake 3 — Assuming a Narrower CI Always Means "Better" Data (좁은 신뢰구간이 항상 "더 나은" 데이터라고 착각)
A narrow CI just means low variance and/or a large sample — it says nothing about whether the sample was representative. A narrow CI built from a biased sample (e.g., only surveying your most loyal customers) can be confidently, precisely wrong.  
좁은 신뢰구간은 그저 낮은 분산이나 큰 표본을 의미할 뿐, 표본이 대표성이 있는지에 대해서는 아무것도 말해주지 않습니다 -- 편향된 표본(예: 가장 충성도 높은 고객만 설문)에서 만들어진 좁은 신뢰구간은 자신 있게, 정밀하게 틀릴 수 있습니다.

✅ **Fix / 해결:** Check sampling methodology for bias BEFORE trusting a narrow interval — width alone doesn't validate the sample.

### Mistake 4 — Forgetting Standard Error Needs the SAMPLE Standard Deviation (표준오차에 표본표준편차가 필요하다는 것을 잊음)
Standard error is `sample_std / sqrt(n)`, and that std dev should be the SAMPLE version (`ddof=1`), continuing the exact numpy-vs-pandas default mismatch from Section 1 — getting this wrong shifts every confidence interval you compute.  
표준오차는 표본표준편차/sqrt(n)이며, 그 표준편차는 표본 버전(ddof=1)이어야 합니다 -- 섹션 1에서 다룬 numpy와 pandas의 기본값 불일치와 정확히 연결됩니다. 이를 틀리면 계산하는 모든 신뢰구간이 어긋납니다.

✅ **Fix / 해결:** When using `numpy`, always pass `ddof=1` explicitly for standard error calculations; `pandas`'s `.std()` already defaults to this.

### Mistake 5 — Expecting a Bigger Sample to Massively Shrink the Interval (표본을 늘리면 구간이 크게 줄어들 거라 기대)
Standard error shrinks with the SQUARE ROOT of sample size — quadrupling your sample only halves your margin of error, and going from 1,000 to 10,000 respondents barely moves the needle. Diminishing returns set in fast (see Example 1).  
표준오차는 표본 크기의 제곱근에 반비례해 줄어듭니다 -- 표본을 4배로 늘려도 오차범위는 절반만 줄어들고, 1,000명에서 10,000명으로 늘려도 거의 변화가 없습니다. 수확체감이 빠르게 나타납니다(예제 1 참고).

✅ **Fix / 해결:** Before collecting more data purely to narrow a CI, calculate the expected new standard error — the cost/benefit often doesn't justify it.

### Mistake 6 — Confusing a Confidence Interval with a Prediction Interval (신뢰구간과 예측구간 혼동)
A confidence interval estimates where the population MEAN likely falls; a prediction interval (a different, wider calculation) estimates where a single NEW data point is likely to fall. Using a CI to predict one customer's individual order amount will be overconfident and too narrow.  
신뢰구간은 모집단 평균이 있을 만한 위치를 추정하고, 예측구간(다른, 더 넓은 계산)은 하나의 새로운 데이터 포인트가 있을 만한 위치를 추정합니다. 신뢰구간으로 한 고객의 개별 주문금액을 예측하면 과신하게 되고 지나치게 좁습니다.

✅ **Fix / 해결:** Use a CI when the question is about an average/aggregate; use a (wider) prediction interval when the question is about a single future observation.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `stats.t.interval()` needs THREE things: confidence level, degrees of freedom (n-1), and the sample mean/SE as loc/scale — always pass them as keyword arguments to avoid silently mixing up the order.<br>stats.t.interval()은 신뢰수준, 자유도(n-1), 표본평균/SE(loc/scale) 세 가지가 필요합니다 -- 인자 순서를 헷갈리지 않도록 항상 키워드 인자로 넘기세요.
- A 90% CI is narrower than a 95% CI, which is narrower than a 99% CI — higher confidence always costs you a wider (less precise) interval; there's no free lunch.<br>90% 신뢰구간은 95%보다 좁고, 95%는 99%보다 좁습니다 -- 높은 신뢰수준은 항상 더 넓은(덜 정밀한) 구간이라는 대가를 치릅니다. 공짜는 없습니다.
- For a quick, rough proportion CI without scipy, the shortcut `± 1/sqrt(n)` (based on a worst-case p=0.5) gives a conservative estimate — useful for a fast gut-check before running the exact formula.<br>scipy 없이 빠르고 대략적인 비율 신뢰구간이 필요하면, ±1/sqrt(n)(최악의 경우 p=0.5 가정) 지름길이 보수적인 추정치를 줍니다 -- 정확한 공식을 돌리기 전 빠른 직감 확인에 유용합니다.
- When a CI for a proportion is being calculated near 0% or 100% (very rare or very common events), the standard formula can behave oddly — non-parametric alternatives later in this curriculum become useful there.<br>비율의 신뢰구간이 0%나 100% 근처(매우 드물거나 매우 흔한 사건)에서 계산될 때는 표준 공식이 이상하게 동작할 수 있습니다 -- 이 커리큘럼 후반의 비모수적 대안이 이런 상황에서 유용해집니다.
- Whenever you report a point estimate in a business document, make it a habit to add the confidence interval right next to it in parentheses — it costs one extra sentence and immediately signals statistical maturity.<br>비즈니스 문서에 점추정치를 보고할 때는 옆에 괄호로 신뢰구간을 추가하는 습관을 들이세요 -- 문장 하나 더 쓰는 비용으로 통계적 성숙도를 즉시 보여줄 수 있습니다.

---
# 🔗 Related Concepts (관련 개념)

```
Probability Distributions (Section 3)
   Normal / Binomial / Poisson, Central Limit Theorem
        ↓
Sampling & Confidence Intervals (Section 4)  ← you are here / 현재 위치
   standard error, CI for means, CI for proportions, correct interpretation
        ↓
Hypothesis Testing Fundamentals (Section 6)
   p-values, significance, Type I/II errors
        ↓
t-Tests (Section 7) -> ... -> A/B Testing (later sections)
```

*How is today's topic connected to other concepts?*

**EN:** Every formula in this section is the Central Limit Theorem from Section 3 put to work — standard error only makes sense because the sample mean's distribution is approximately Normal. The exact machinery you built today (the t-distribution CI for a mean, the proportion CI) reappears almost unchanged as the confidence-interval half of every t-test's output starting in Section 7, and the proportion CI is the direct mathematical backbone of comparing two A/B test variants.

**KR:** 이 섹션의 모든 공식은 섹션 3의 중심극한정리를 실제로 활용한 것입니다 -- 표본평균의 분포가 대략 정규분포이기 때문에 표준오차가 의미를 가집니다. 오늘 만든 정확한 도구들(평균의 t-분포 신뢰구간, 비율의 신뢰구간)은 섹션 7부터 모든 t-검정 출력의 신뢰구간 부분으로 거의 그대로 다시 등장하며, 비율의 신뢰구간은 A/B 테스트의 두 변형을 비교하는 직접적인 수학적 근간입니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** You're preparing a monthly business review. Leadership wants two numbers presented with appropriate uncertainty: average customer lifetime value (a mean) and the trial-to-paid conversion rate (a proportion), both from this month's cohort of 250 new customers.

**KR:** 월간 비즈니스 리뷰를 준비하고 있습니다. 경영진은 적절한 불확실성과 함께 두 숫자를 원합니다: 평균 고객 생애가치(평균)와 무료체험-유료전환율(비율), 둘 다 이번 달 신규 고객 250명 코호트에서.

**To-do / 할 일:**
- [x] Compute the mean LTV with a 95% confidence interval<br>95% 신뢰구간과 함께 평균 LTV 계산하기
- [x] Compute the conversion rate with a 95% confidence interval<br>95% 신뢰구간과 함께 전환율 계산하기
- [x] Decide whether either interval is too wide to be useful for planning<br>둘 중 계획에 쓰기엔 너무 넓은 구간이 있는지 판단하기
- [x] Write both results in the format leadership can put directly into a slide<br>경영진이 슬라이드에 바로 넣을 수 있는 형식으로 두 결과 작성하기

In [8]:
np.random.seed(6)
n_cohort = 250
ltv = np.random.gamma(shape=2.5, scale=80, size=n_cohort)
converted = np.random.random(n_cohort) < 0.14  # true underlying rate is 14%

# CI for mean LTV
mean_ltv = ltv.mean()
se_ltv = ltv.std(ddof=1) / np.sqrt(n_cohort)
ci_ltv = stats.t.interval(0.95, df=n_cohort - 1, loc=mean_ltv, scale=se_ltv)

# CI for conversion proportion
p_hat = converted.mean()
se_p = np.sqrt(p_hat * (1 - p_hat) / n_cohort)
z_crit = stats.norm.ppf(0.975)
ci_p = (p_hat - z_crit * se_p, p_hat + z_crit * se_p)

print(f"Average LTV: ${mean_ltv:.0f}, 95% CI: [${ci_ltv[0]:.0f}, ${ci_ltv[1]:.0f}]")
print(f"Conversion rate: {p_hat:.1%}, 95% CI: [{ci_p[0]:.1%}, {ci_p[1]:.1%}]")

ltv_width_pct = (ci_ltv[1] - ci_ltv[0]) / mean_ltv * 100
p_width_pct = (ci_p[1] - ci_p[0]) / p_hat * 100
print(f"\nLTV interval width as % of the estimate: {ltv_width_pct:.0f}%")
print(f"Conversion rate interval width as % of the estimate: {p_width_pct:.0f}%")

if p_width_pct > 30:
    print("\nThe conversion rate CI is quite wide relative to the estimate -- consider")
    print("waiting for a larger cohort before making firm budget decisions on it.")

print()
print(f"평균 LTV: ${mean_ltv:.0f}, 95% CI: [${ci_ltv[0]:.0f}, ${ci_ltv[1]:.0f}]")
print(f"전환율: {p_hat:.1%}, 95% CI: [{ci_p[0]:.1%}, {ci_p[1]:.1%}]")
print(f"LTV 구간 폭(추정치 대비 %): {ltv_width_pct:.0f}%")
print(f"전환율 구간 폭(추정치 대비 %): {p_width_pct:.0f}%")
if p_width_pct > 30:
    print("전환율 신뢰구간이 추정치 대비 상당히 넓습니다 -- 예산에 관한 확정적인")
    print("결정을 내리기 전에 더 큰 코호트를 기다리는 것을 고려하세요.")

Average LTV: $197, 95% CI: [$182, $212]
Conversion rate: 12.0%, 95% CI: [8.0%, 16.0%]

LTV interval width as % of the estimate: 15%
Conversion rate interval width as % of the estimate: 67%

The conversion rate CI is quite wide relative to the estimate -- consider
waiting for a larger cohort before making firm budget decisions on it.

평균 LTV: $197, 95% CI: [$182, $212]
전환율: 12.0%, 95% CI: [8.0%, 16.0%]
LTV 구간 폭(추정치 대비 %): 15%
전환율 구간 폭(추정치 대비 %): 67%
전환율 신뢰구간이 추정치 대비 상당히 넓습니다 -- 예산에 관한 확정적인
결정을 내리기 전에 더 큰 코호트를 기다리는 것을 고려하세요.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** A sample statistic varies from sample to sample, and standard error quantifies exactly how much — shrinking with the square root of sample size, which means diminishing returns kick in fast as you collect more data. A confidence interval for a mean uses the t-distribution (which has fatter tails than Normal to account for the extra uncertainty of estimating standard deviation from a sample), while a confidence interval for a proportion uses the Normal distribution directly, since a proportion is really just the mean of a series of 0/1 outcomes. The single most commonly misunderstood part of this topic is what "95% confidence" actually means: it's not a 95% probability that the true value sits inside any one specific interval, but rather a statement about how often the CI-building PROCEDURE succeeds if repeated many times — a distinction Example 4's simulation makes concrete.

**KR:** 표본 통계량은 표본마다 다르며, 표준오차는 그 정도를 정확히 정량화합니다 -- 표본 크기의 제곱근에 반비례해 줄어들므로, 데이터를 더 모을수록 수확체감이 빠르게 나타납니다. 평균의 신뢰구간은 t-분포를 사용하고(표본에서 표준편차를 추정하는 추가적인 불확실성을 반영하기 위해 정규분포보다 두꺼운 꼬리를 가짐), 비율의 신뢰구간은 정규분포를 직접 사용합니다(비율은 사실 0/1 결과들의 평균이기 때문). 이 주제에서 가장 흔히 오해되는 부분은 "95% 신뢰"가 실제로 무엇을 의미하는지입니다: 실제 값이 특정 하나의 구간 안에 있을 확률이 95%라는 뜻이 아니라, 신뢰구간을 만드는 절차를 여러 번 반복했을 때 얼마나 자주 성공하는지에 대한 진술입니다 -- 예제 4의 시뮬레이션이 이를 구체적으로 보여줍니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** Standard error measures how much a sample statistic would vary across repeated samples (shrinking with the square root of n), and a confidence interval uses that number to report an honest range instead of a false-precision point estimate — with "95% confidence" describing the long-run reliability of the method, not the probability that one specific interval happens to be correct.

> **KR:** 표준오차는 표본 통계량이 반복된 표본에서 얼마나 달라질지를 측정하고(n의 제곱근에 반비례해 줄어듦), 신뢰구간은 그 숫자를 사용해 거짓된 정밀도의 점추정치 대신 정직한 범위를 보고합니다 -- "95% 신뢰"는 하나의 특정 구간이 우연히 맞을 확률이 아니라, 그 방법의 장기적 신뢰도를 설명합니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** If you quadruple your sample size, what happens to your standard error, and why isn't it cut to a quarter?  
**Q1 (KR).** 표본 크기를 4배로 늘리면 표준오차는 어떻게 되며, 왜 4분의 1로 줄지 않나요?

The standard error is cut in half, not to a quarter.  
표준오차는 s/√n이라, 표본을 4배로 늘리면 √4 = 2가 분모에 들어갑니다. 그래서 절반이 됩니다.

**Q2 (EN).** Why does a confidence interval for a mean use the t-distribution instead of the Normal distribution, especially for smaller samples?  
**Q2 (KR).** 평균의 신뢰구간은 왜, 특히 작은 표본에서, 정규분포 대신 t-분포를 사용하나요?

The population standard deviation is unknown, so a small sample's standard deviation is itself uncertain.  
모표준편차를 표본으로 추정해야 해서, 작은 표본에서는 꼬리가 더 두꺼운 t-분포를 씁니다. 정규분포를 쓰면 구간이 너무 좁아집니다.

**Q3 (EN).** What's technically wrong with saying "there's a 95% probability the true mean is in this interval," and what should you say instead?  
**Q3 (KR).** "실제 평균이 이 구간에 있을 확률이 95%다"라는 말의 어디가 기술적으로 틀렸으며, 대신 뭐라고 말해야 하나요?

The true mean is fixed. It is not a random 95% chance sitting inside this one interval.  
실제 평균은 고정되어 있습니다. "같은 방식으로 표본을 많이 뽑으면, 만든 구간 중 약 95%가 실제 평균을 포함한다"고 말합니다.

**Q4 (EN).** A colleague reports a very narrow confidence interval and calls the result "precise." What question should you ask before trusting that claim?  
**Q4 (KR).** 동료가 매우 좁은 신뢰구간을 보고하며 결과가 "정밀하다"고 말합니다. 그 주장을 믿기 전에 어떤 질문을 해야 하나요?

Ask whether the sample was representative, or only a biased slice of customers.  
표본이 전체를 대표하는지 물어봅니다. 좁은 구간은 분산이 작거나 표본이 크다는 뜻이지, 편향된 표본의 오답을 고쳐 주지는 않습니다.

**Q5 (EN).** What's the difference between a confidence interval and a prediction interval, and which one would you use to estimate where a single new customer's order amount will fall?  
**Q5 (KR).** 신뢰구간과 예측구간의 차이는 무엇이며, 한 명의 신규 고객의 주문금액이 어디쯤 될지 추정하려면 어느 것을 사용해야 하나요?

A confidence interval estimates the population mean. A prediction interval estimates one new observation, and it is wider.  
한 명의 신규 고객 주문금액은 예측구간을 씁니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*